## Практическое задание №1
### по дисциплине "Введение в анализ данных"

*Работу выполнил*: Евсеев Григорий, 3826М1ПМад1


Данная работа выполнена с использованием IDE Visual Studio Code в формате Jupyter Notebook. Ядро Python 3.12.3

## Часть 1.

Сведения о таблице:

In [1]:
import xlrd

book = xlrd.open_workbook('01.xlsx')
sheet = book.sheet_by_name('Отчет')

num_rows = sheet.nrows
num_cols = sheet.ncols

print(' Количество листов = ', book.nsheets, '\n Количество строк = ', num_rows, '\n Количество столбцов = ', num_cols)

 Количество листов =  2 
 Количество строк =  101 
 Количество столбцов =  5


Список годов, в которых количество обучающихся было максимальным:

In [2]:
import numpy as np

regionCount = []

for i in range(num_rows - 4):
    if (sheet.cell(i + 4, 0).value.find('федеральный округ') == -1):
        regionCount.append(sheet.row_values(i + 4, 1, 4))

yearMax = 2016 + np.argmax(regionCount, axis = 1)

print(yearMax)

[2017 2017 2017 2018 2018 2017 2018 2017 2017 2016 2016 2018 2016 2018
 2017 2016 2016 2016 2016 2016 2018 2018 2017 2016 2016 2016 2016 2017
 2016 2018 2016 2016 2017 2018 2018 2016 2016 2016 2016 2018 2016 2016
 2016 2016 2017 2016 2016 2016 2016 2016 2016 2016 2016 2017 2016 2016
 2018 2016 2018 2018 2016 2017 2016 2016 2016 2016 2016 2016 2016 2018
 2018 2016 2016 2016 2016 2017 2016 2016 2016 2017 2018 2018 2017 2016
 2016 2016 2016]


Число субъектов РФ, в которых именно в этом году число обучающихся было максимальным:

In [3]:
from collections import Counter

yearCounter = Counter(yearMax)
print(yearCounter)

Counter({np.int64(2016): 53, np.int64(2018): 18, np.int64(2017): 16})


Сортировка субъектов РФ по возрастанию среднего количества обучающихся за 3 года:

In [4]:
regionMean = {}

for i in range(num_rows - 4):
    cell_value = sheet.cell(i + 4, 0).value
    if (cell_value.find('федеральный округ') != -1):
        continue
    row_vals = sheet.row_values(i + 4, 1, 4)
    numeric_vals = [float(x) for x in row_vals]
    regionMean[cell_value] = np.mean(numeric_vals)

def secondInPair(x):
    (a, b) = x
    return b

regionSorted = sorted(regionMean.items(), key=secondInPair)
print(regionSorted)

[('Алтайский край', np.float64(333342782.33)), ('Краснодарский край', np.float64(1000025179.3299999)), ('Красноярский край', np.float64(1333354935.6633332)), ('Приморский край', np.float64(1666672139.0)), ('Ставропольский край', np.float64(2333345888.0)), ('Хабаровский край', np.float64(2666679640.9966664)), ('Амурская область', np.float64(3333339395.0)), ('Архангельская область', np.float64(3666674886.3266664)), ('Архангельская область (кроме Ненецкого автономного округа)', np.float64(3667007975.66)), ('Ненецкий автономный округ (Архангельская область)', np.float64(3700000244.0)), ('Астраханская область', np.float64(4000002737.3333335)), ('Белгородская область', np.float64(4666680370.666667)), ('Брянская область', np.float64(5000003181.0)), ('Владимирская область', np.float64(5666671725.663333)), ('Волгоградская область', np.float64(6000008126.330001)), ('Вологодская область', np.float64(6333343679.333333)), ('Воронежская область', np.float64(6666679667.996667)), ('Нижегородская облас

## Часть 2.

### Пункт a: Загрузка данных из документов для работы в Python

In [5]:
import xlrd
import numpy as np
from collections import Counter

Открытие нужных книг и листов "Отчет". Вывод данных о количестве листов, строк и столбцов.

In [6]:
book1 = xlrd.open_workbook('01_Образование_организации_01.xlsx') # загрузка книги
book2 = xlrd.open_workbook('01_Образование_организации_02.xlsx')

sheet1 = book1.sheet_by_name('Отчет')   # загрузка листа
sheet2 = book2.sheet_by_name('Отчет')

num_rows1 = sheet1.nrows    # количество строк
num_cols1 = sheet1.ncols    # количество столбцов

num_rows2 = sheet2.nrows
num_cols2 = sheet2.ncols

# вывод
print('Книга 1')
print(' Количество листов = ', book1.nsheets, '\n Количество строк = ', num_rows1, '\n Количество столбцов = ', num_cols1)
print('\n')
print('Книга 2')
print(' Количество листов = ', book2.nsheets, '\n Количество строк = ', num_rows2, '\n Количество столбцов = ', num_cols2)

Книга 1
 Количество листов =  2 
 Количество строк =  98 
 Количество столбцов =  3


Книга 2
 Количество листов =  2 
 Количество строк =  104 
 Количество столбцов =  3


### Пункт b. Составление словаря (ключ - "название субъекта РФ")

Функция `is_valid_region` возвращает `True`, если название строки является субъектом РФ; `False` в иных случаях (например, сводная запись)

In [7]:
def is_valid_region(name):
    name_lower = name.lower()
    if (('федеральный округ' in name_lower) or ('российская федерация' in name_lower)):
        return False
    
    return True

Функция `parse_value` преобразует число в ячейках в тип `float`; если ячейка пустая или содержит не число, то возвращает `None`

In [8]:
def parse_value(cell_value):
    if ((cell_value == '') or (cell_value is None)):
        return None
    try:
        return float(cell_value)
    except Exception:
        return None

Составление словаря, в котором ключом является субъект РФ, а значением список из 4 чисел: количество образовательных организаций в 2015 - 2018 годах соответственно:

In [9]:
data = {}       # инициализация пустого словаря


# файл 1, в котором данные за 2016 и 2017 годы
for i in range(3, sheet1.nrows):                # первые 3 строки содержат неинформативные данные
    name = sheet1.cell_value(i, 0).strip()      # извлечение названия региона из ячейки (А; i) 
    if not name or not is_valid_region(name):   # пропуск пустых ячеек или с нечисловым значением
        continue
    val_2016 = parse_value(sheet1.cell_value(i, 1)) # значения из столбца В
    val_2017 = parse_value(sheet1.cell_value(i, 2)) # значения из столбца C
    data[name] = [None, val_2016, val_2017, None]   # добавление записей с "подготовкой" к сортировке


# файл 2, в котором данные за 2015 и 2018 годы
for i in range(3, sheet2.nrows):
    name = sheet2.cell_value(i, 0).strip()
    if not name or not is_valid_region(name):
        continue
    val_2015 = parse_value(sheet2.cell_value(i, 1))
    val_2018 = parse_value(sheet2.cell_value(i, 2))
    if name in data:                                # проверка, есть ли уже субъект РФ в словаре; если есть, добавление данных в позиции 0 и 3, соотвеющих
        data[name][0] = val_2015                    # 2015 и 2018 годам
        data[name][3] = val_2018
    else:
        data[name] = [val_2015, None, None, val_2018]   # если нет, то создается новая запись только с этими годами

# вывод
print(f"Всего субъектов: {len(data)}")
print(data)


Всего субъектов: 89
{'Белгородская область': [193.0, 906.0, 916.0, 885.0], 'Брянская область': [163.0, 607.0, 715.0, 727.0], 'Владимирская область': [127.0, 137.0, 488.0, 554.0], 'Воронежская область': [210.0, 521.0, 1106.0, 1130.0], 'Ивановская область': [102.0, 433.0, 437.0, 449.0], 'Калужская область': [113.0, 280.0, 376.0, 384.0], 'Костромская область': [109.0, 303.0, 352.0, 348.0], 'Курская область': [118.0, 527.0, 598.0, 607.0], 'Липецкая область': [122.0, 490.0, 515.0, 432.0], 'Московская область': [460.0, 2779.0, 3164.0, 3235.0], 'Орловская область': [136.0, 500.0, 526.0, 483.0], 'Рязанская область': [191.0, 315.0, 466.0, 483.0], 'Смоленская область': [136.0, 613.0, 670.0, 671.0], 'Тамбовская область': [98.0, 687.0, 680.0, 670.0], 'Тверская область': [165.0, 428.0, 587.0, 702.0], 'Тульская область': [96.0, 630.0, 704.0, 693.0], 'Ярославская область': [190.0, 348.0, 433.0, 421.0], 'Город Москва столица Российской Федерации город федерального значения': [240.0, 935.0, 1040.0, 943

### Пункт c. Добавление среднего количества образовательных организаций за 4 года и год, в котором было достигнуто максимальное количество образовательных организаций

Очистка словаря от пропущенных значений

In [10]:
data_clean = {k: v for k, v in data.items() if None not in v}

Вычисление среднего и нахождение максимума. Для примера, выведены три области из итогового словаря.

In [11]:
years = [2015, 2016, 2017, 2018]
for name, values in data_clean.items():
    avg = np.mean(values)       # расчет среднего
    max_val = max(values)       # расчет максимального числа организаций
    max_year = years[values.index(max_val)]     # расчет года, в котором был достигнут максмум
    values.append(avg)      # добавление среднего
    values.append(max_year) # добавление в конец словаря года, в котором был максимум

# пример вывода
print(data_clean['Белгородская область'])
print(data_clean['Брянская область'])
print(data_clean['Нижегородская область'])

[193.0, 906.0, 916.0, 885.0, np.float64(725.0), 2017]
[163.0, 607.0, 715.0, 727.0, np.float64(553.0), 2018]
[388.0, 1575.0, 1617.0, 1640.0, np.float64(1305.0), 2018]


### Пункт d. Составление словаря годов максимумов

Сбор годов, в которых был достигнут максимум:

In [12]:
max_years = [values[5] for values in data_clean.values()]
print(max_years)

[2017, 2018, 2018, 2018, 2018, 2018, 2017, 2018, 2017, 2018, 2017, 2018, 2018, 2016, 2018, 2017, 2017, 2017, 2018, 2017, 2018, 2016, 2018, 2017, 2017, 2017, 2017, 2018, 2018, 2017, 2018, 2018, 2017, 2017, 2017, 2018, 2017, 2018, 2017, 2017, 2018, 2017, 2018, 2018, 2018, 2016, 2017, 2018, 2017, 2018, 2017, 2017, 2018, 2018, 2017, 2018, 2018, 2018, 2018, 2018, 2018, 2018, 2018, 2018, 2018, 2016, 2018, 2018, 2018, 2017, 2018, 2018, 2018, 2018, 2018, 2018, 2018, 2017, 2018, 2018, 2018, 2017, 2018, 2018, 2016]


С применением `Counter` находится количество субъектов РФ, в которых число организаций достигло максимума в каждом из годов:

In [13]:
year_counter = Counter(max_years)
print(year_counter)

Counter({2018: 52, 2017: 28, 2016: 5})


### Пункт е. Сортировка субъектов по возрастанию среднего числа организаций за 4 года

Функция для расчета среднего, где `item[1]` - список значений для субъекта (годы, среднее, максимум); `item[1][4]` - среднее количество организаций за 4 года (искомое)

In [14]:
def get_average(item):
    return item[1][4]       

С помощью функции `sorted` происходит сортировка очищенной выборки по среднему значению (ключ)

In [15]:
sorted_regions = sorted(data_clean.items(), key=get_average)
sorted_regions

[('Ненецкий автономный округ (Архангельская область)',
  [10.0, 41.0, 30.0, 32.0, np.float64(28.25), 2016]),
 ('Чукотский автономный округ',
  [8.0, 45.0, 44.0, 44.0, np.float64(35.25), 2016]),
 ('Город федерального значения Севастополь',
  [20.0, 66.0, 79.0, 46.0, np.float64(52.75), 2017]),
 ('Республика Адыгея (Адыгея)',
  [59.0, 49.0, 61.0, 44.0, np.float64(53.25), 2017]),
 ('Республика Калмыкия', [38.0, 42.0, 48.0, 108.0, np.float64(59.0), 2018]),
 ('Еврейская автономная область',
  [22.0, 75.0, 84.0, 89.0, np.float64(67.5), 2018]),
 ('Магаданская область', [39.0, 77.0, 86.0, 81.0, np.float64(70.75), 2017]),
 ('Республика Ингушетия*', [15.0, 19.0, 156.0, 136.0, np.float64(81.5), 2017]),
 ('Республика Северная Осетия-Алания',
  [94.0, 105.0, 172.0, 137.0, np.float64(127.0), 2017]),
 ('Республика Мордовия',
  [115.0, 153.0, 177.0, 166.0, np.float64(152.75), 2017]),
 ('Камчатский край', [91.0, 174.0, 182.0, 175.0, np.float64(155.5), 2017]),
 ('Карачаево-Черкесская Республика',
  [73.0